<a href="https://colab.research.google.com/github/arnavon2005/Army_Provost_ML_Project/blob/main/06_Decision_Support_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Decision-Support System

This notebook develops the decision-support layer of the Army Provost ML project. It builds upon the completed exploratory analysis and machine-learning model developed in the previous notebooks.

The objective is to transform incident-level predictions and historical crime patterns into interpretable information that can assist control-room personnel with incident assessment, prioritization, and response planning.

The system is a decision-support prototype. It does not autonomously make operational decisions or claim to represent official Indian Army doctrine.

In [1]:
# ============================================================
# DECISION-SUPPORT SYSTEM
# PROJECT ENVIRONMENT SETUP
# ============================================================

from google.colab import drive
import os

print("=" * 70)
print("ARMY PROVOST DECISION-SUPPORT SYSTEM")
print("PROJECT ENVIRONMENT SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# Existing project root
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"

DATASETS_PATH = os.path.join(
    PROJECT_ROOT,
    "Datasets"
)

CLEANED_DATA_PATH = os.path.join(
    DATASETS_PATH,
    "Cleaned"
)

MODELS_PATH = os.path.join(
    PROJECT_ROOT,
    "Models"
)

FIGURES_PATH = os.path.join(
    PROJECT_ROOT,
    "Figures"
)

OUTPUTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs"
)

REPORTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Reports"
)

LOGS_PATH = os.path.join(
    PROJECT_ROOT,
    "Logs"
)

# ------------------------------------------------------------
# Important saved ML artifacts
# ------------------------------------------------------------

RF_MODEL_PATH = os.path.join(
    MODELS_PATH,
    "final_random_forest.pkl"
)

PREPROCESSING_PIPELINE_PATH = os.path.join(
    MODELS_PATH,
    "preprocessing_pipeline.pkl"
)

# ------------------------------------------------------------
# Verify project structure
# ------------------------------------------------------------

required_directories = [
    PROJECT_ROOT,
    DATASETS_PATH,
    CLEANED_DATA_PATH,
    MODELS_PATH,
    FIGURES_PATH,
    OUTPUTS_PATH,
    REPORTS_PATH,
    LOGS_PATH
]

print("\nChecking project directories...\n")

all_directories_available = True

for path in required_directories:
    exists = os.path.isdir(path)

    print(
        f"{'✓' if exists else '✗'} "
        f"{path}"
    )

    if not exists:
        all_directories_available = False

# ------------------------------------------------------------
# Verify important ML artifacts
# ------------------------------------------------------------

print("\nChecking important ML artifacts...\n")

required_files = [
    RF_MODEL_PATH,
    PREPROCESSING_PIPELINE_PATH,
    os.path.join(
        OUTPUTS_PATH,
        "Final_Random_Forest_Diagnostic_Summary.csv"
    ),
    os.path.join(
        OUTPUTS_PATH,
        "Final_Model_Performance_Comparison.csv"
    )
]

all_files_available = True

for path in required_files:
    exists = os.path.isfile(path)

    print(
        f"{'✓' if exists else '✗'} "
        f"{os.path.basename(path)}"
    )

    if not exists:
        all_files_available = False

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if all_directories_available and all_files_available:
    print("PROJECT ENVIRONMENT VERIFIED SUCCESSFULLY")
    print("Ready to begin the Decision-Support System phase.")
else:
    print("PROJECT ENVIRONMENT VERIFICATION INCOMPLETE")
    print("Review the missing paths/files above before proceeding.")

print("=" * 70)

print("\nProject Root:")
print(PROJECT_ROOT)

ARMY PROVOST DECISION-SUPPORT SYSTEM
PROJECT ENVIRONMENT SETUP
Mounted at /content/drive

Checking project directories...

✓ /content/drive/MyDrive/Army_Provost_ML_Project
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Datasets
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Datasets/Cleaned
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Models
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Figures
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Outputs
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Reports
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Logs

Checking important ML artifacts...

✓ final_random_forest.pkl
✓ preprocessing_pipeline.pkl
✓ Final_Random_Forest_Diagnostic_Summary.csv
✓ Final_Model_Performance_Comparison.csv

PROJECT ENVIRONMENT VERIFIED SUCCESSFULLY
Ready to begin the Decision-Support System phase.

Project Root:
/content/drive/MyDrive/Army_Provost_ML_Project


In [2]:
# ============================================================
# PRIMARY INCIDENT CATEGORY INVENTORY
# DECISION-SUPPORT SYSTEM
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("PRIMARY INCIDENT CATEGORY INVENTORY")
print("=" * 70)

# ------------------------------------------------------------
# Existing project paths
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"

CLEANED_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "Datasets",
    "Cleaned"
)

CLEANED_FILE = os.path.join(
    CLEANED_DATA_PATH,
    "chicago_crimes_cleaned.csv"
)

# ------------------------------------------------------------
# Verify dataset exists
# ------------------------------------------------------------

if not os.path.isfile(CLEANED_FILE):
    raise FileNotFoundError(
        f"Cleaned dataset not found:\n{CLEANED_FILE}"
    )

print("\nDataset found:")
print(CLEANED_FILE)

# ------------------------------------------------------------
# Read ONLY Primary Type
# ------------------------------------------------------------

print("\nReading only the 'Primary Type' column...")
print("The full dataset will NOT be loaded into memory.")

primary_type_series = pd.read_csv(
    CLEANED_FILE,
    usecols=["Primary Type"]
)["Primary Type"]

# ------------------------------------------------------------
# Generate category inventory
# ------------------------------------------------------------

primary_type_counts = (
    primary_type_series
    .value_counts(dropna=False)
    .reset_index()
)

primary_type_counts.columns = [
    "Primary Type",
    "Incident Count"
]

primary_type_counts["Percentage"] = (
    primary_type_counts["Incident Count"]
    / primary_type_counts["Incident Count"].sum()
    * 100
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATEGORY INVENTORY")
print("=" * 70)

print(
    f"\nTotal records inspected: "
    f"{len(primary_type_series):,}"
)

print(
    f"Unique Primary Type categories: "
    f"{primary_type_series.nunique(dropna=True)}"
)

print("\nComplete Primary Type Distribution:\n")

display(
    primary_type_counts.round({
        "Percentage": 2
    })
)

# ------------------------------------------------------------
# Save inventory permanently
# ------------------------------------------------------------

OUTPUTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs"
)

taxonomy_inventory_path = os.path.join(
    OUTPUTS_PATH,
    "Primary_Type_Category_Inventory.csv"
)

primary_type_counts.to_csv(
    taxonomy_inventory_path,
    index=False
)

print("=" * 70)
print("CATEGORY INVENTORY SAVED")
print("=" * 70)

print("\nSaved to:")
print(taxonomy_inventory_path)

PRIMARY INCIDENT CATEGORY INVENTORY

Dataset found:
/content/drive/MyDrive/Army_Provost_ML_Project/Datasets/Cleaned/chicago_crimes_cleaned.csv

Reading only the 'Primary Type' column...
The full dataset will NOT be loaded into memory.

CATEGORY INVENTORY

Total records inspected: 8,602,734
Unique Primary Type categories: 34

Complete Primary Type Distribution:



,Primary Type,Incident Count,Percentage
0,THEFT,1827377,21.24
1,BATTERY,1567212,18.22
2,CRIMINAL DAMAGE,977357,11.36
3,NARCOTICS,768704,8.94
4,ASSAULT,580094,6.74
5,OTHER OFFENSE,537540,6.25
6,BURGLARY,455436,5.29
7,MOTOR VEHICLE THEFT,444725,5.17
8,DECEPTIVE PRACTICE,400071,4.65
9,ROBBERY,318103,3.70


CATEGORY INVENTORY SAVED

Saved to:
/content/drive/MyDrive/Army_Provost_ML_Project/Outputs/Primary_Type_Category_Inventory.csv
